In [2]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber

In [3]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [4]:
def load_keywords(json_file="quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [5]:
def extract_statement(exercise):
    m = re.search("R E S O L U C I Ó N", exercise)
    if m:
        end_pos = m.start()
        exercise = exercise[:end_pos].strip()
        lines = exercise.split("\n")
        exam = lines[-1]
        statement = "\n".join(lines[:-1])
        return exam, statement
    return None

In [6]:
def extract_exercises(file):
    exercises = []
    with pdfplumber.open(file) as pdf:
        for idx, page in enumerate(pdf.pages):
            texto = ext.extract_text_from_page(page)
            if idx > 0:  # ejercicios comienzan en 2da página
                st = extract_statement(texto)
                if st:
                    exercises.append(extract_statement(texto))
    return exercises

In [17]:
def classify_exercise(statement: str, topic: str, keywords = None) -> tuple:
    if keywords is None:
        keywords = load_keywords()

    topic_keywords = keywords[topic]["keywords_tema"]
    
    pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
    if re.search(pattern, statement):
        exercise_topic = topic
    else:
        exercise_topic = None

    if exercise_topic:
        exercise_types = []
        exercise_type_keywords = keywords[topic]["tipos_ejercicio"]
        
        for tipo, tipo_keywords in exercise_type_keywords.items():
            pattern = re.compile("|".join(tipo_keywords), flags=re.IGNORECASE | re.DOTALL)
            if re.search(pattern, statement):
                exercise_types.append(tipo)
    else:
        exercise_types = [] 

    return exercise_topic, exercise_types

In [7]:
def create_df_exercises(folder: Path) -> pd.DataFrame:
    exercises_list = []
    if not isinstance(folder, Path):
        raise ValueError("La función espera un objeto Path con el directorio a procesar.")
        if not folder.is_dir():
            raise ValueError("El objeto Path debe ser un directorio.")
    
    for file in folder.iterdir():   
        exams, statements = [], []
        #print("Processing file: ", file.name)
        exercises = extract_exercises(file)
        for ex in exercises:
            if ex:
                exercises_list.append({"exam" : ex[0], "statement": ex[1]})
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de una carpeta

In [9]:
# # RENOMBRAR CARPETAS PARA QUE COINCIDAN CON CLAVES DEL JSON CON KEYWORDS
# import shutil

# acentos = {"á" : 'a', "é" : 'e', 'í' : 'i', 'ó' : 'o'}
# translation_table = str.maketrans(acentos)

# for folder in Path("temas").iterdir():
#     new_folder = folder.name.lower().replace(" ", "_")
#     new_folder = new_folder.translate(translation_table)
    
#     new_folder = Path("/".join(folder.parts[:-1]) + f"/{new_folder}")
#     shutil.move(folder, new_folder)

Ordenar directorios sin acentos:

In [10]:
TEMAS = sorted(Path("temas").iterdir())

In [11]:
[i.name for i in TEMAS]

['acido_base',
 'cantidad_quimica',
 'configuracion_electronica',
 'enlace_quimico',
 'equilibrio_quimico',
 'formulacion',
 'reactividad_organica',
 'redox',
 'solubilidad',
 'termoquimica']

## Clasificar carpeta de ejercicios

In [24]:
df = create_df_exercises(TEMAS[3])
df["topic"], df["exercise_type"] = zip(*df.statement.apply(classify_exercise, topic=TEMAS[3].name))
df = df.explode("exercise_type")
df = df.sort_values("exam").reset_index(drop=True)
df = df.drop_duplicates()

In [25]:
df.isna().sum()

exam             0
statement        0
topic            0
exercise_type    0
dtype: int64

In [26]:
for i in df[df.exercise_type.isna()].statement:
    print(i)

In [27]:
df.exercise_type.unique()

<StringArray>
['Solubilidad compuestos covalentes moleculares',
                        'Comparación Born-Haber',
                          'Estructuras de Lewis',
                                   'Hibridación',
                                     'Geometría',
                                'Tipo de enlace',
          'comparación pto. fusión / ebullución',
                                     'Polaridad',
                      'Fuerzas intermoleculares',
                           'Ciclo de Born-Haber',
                           'Hibridación carbono',
     'Propiedades en función del tipo de enlace',
                             'verdadero o falso',
                             'Enlaces múltiples']
Length: 14, dtype: str

In [28]:
for i in df[df.exercise_type == "Hibridación carbono"].statement:
    print(f"{i}\n\n")

a) Escriba las estructuras de Lewis correspondientes a las moléculas de etano (CH₃CH₃) y
eteno (CH₂ =CH₂)
b) Explique qué tipo de hibridación tiene el carbono en cada compuesto.


Razone la veracidad o falsedad de las siguientes afirmaciones:
a) Los metales son buenos conductores de la electricidad.
b) Todos los compuestos de carbono presentan hibridación sp³.
c) Los compuestos iónicos conducen la corriente eléctrica en estado sólido.


a) Dibuje la molécula de eteno (CH =CH ), indicando la hibridación de los átomos de
2 2
carbono y todos los enlaces σ y π presentes.
b) Realice el diagrama de Lewis de la molécula CH Cl.
3
c) Justifique la polaridad de la molécula PH , basándose en la aplicación de la TRPECV
3


Dadas las siguientes moléculas: C H , C H , CH y CH OH.
2 4 2 2 4 3
a) Escriba sus estructuras de Lewis.
b) Indique la hibridación del átomo de carbono en estas moléculas.
c) Justifique cuál de estas moléculas presenta un mayor punto de ebullición.


En referencia al tetrafluoru

In [23]:
hibridacion_carbono = df[df.exercise_type == "Hibridación carbono"]

In [23]:
df_propiedades = df[df.topic.isna()]

In [24]:
df_propiedades["topic"], df_propiedades["exercise_type"] = zip(*df_propiedades.statement.apply(classify_exercise, topic="propiedades_periodicas"))
df_propiedades = df_propiedades.explode("exercise_type")
df_propiedades = df_propiedades.sort_values("exam").reset_index(drop=True)
df_propiedades = df_propiedades.drop_duplicates()

ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
df_propiedades

,exam,statement,topic,exercise_type
0,QUIMICA. 2017. SEPTIEMBRE. EJERCICIO 2 OPCIÓN A,"a) Justifique cuál de las siguientes especies,...",propiedades_periodicas,Propiedades periódicas
1,QUÍMICA. 2000. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Los elementos A y B tienen, en sus últimos niv...",propiedades_periodicas,Propiedades periódicas
2,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,Defina: a) Energía de ionización. b) Afinidad ...,propiedades_periodicas,Propiedades periódicas
3,QUÍMICA. 2004. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,"Considere la serie de elementos: Li, Na, K, Rb...",propiedades_periodicas,Propiedades periódicas
4,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Propiedades periódicas
5,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Comparación energías de ionización
6,QUÍMICA. 2013. JUNIO. EJERCICIO 2. OPCIÓN A,"Para los siguientes elementos Na, P, S y Cl, d...",propiedades_periodicas,Propiedades periódicas
7,QUÍMICA. 2013. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Dado los elementos Cl, K y Ar, ordene razonada...",propiedades_periodicas,Propiedades periódicas
8,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,verdadero o falso
9,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,Propiedades periódicas


In [ ]:
df_propiedades.isna().sum()

exam             0
statement        0
topic            0
exercise_type    0
dtype: int64

In [ ]:
df = pd.merge(df, df_propiedades, on=["exam", "statement"])

In [ ]:
df.isna().sum()

exam             0
statement        0
topic            0
exercise_type    0
dtype: int64

In [ ]:
df["topic"] = df.apply(lambda row: row["topic_y"] if pd.isna(row["topic_x"]) else row["topic_x"], axis=1)
df["exercise_type"] = df.apply(lambda row: row["exercise_type_y"] if pd.isna(row["exercise_type_x"]) else row["exercise_type_x"], axis=1)

In [ ]:
df = df.drop([i for i in df.columns if ("_x" in i) or ("_y" in i)], axis = 1)

In [ ]:
df.sort_values("exam")

,exam,statement,topic,exercise_type
0,QUIMICA. 2017. SEPTIEMBRE. EJERCICIO 2 OPCIÓN A,"a) Justifique cuál de las siguientes especies,...",propiedades_periodicas,Propiedades periódicas
1,QUÍMICA. 2000. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Los elementos A y B tienen, en sus últimos niv...",propiedades_periodicas,Propiedades periódicas
2,QUÍMICA. 2001. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,Defina: a) Energía de ionización. b) Afinidad ...,propiedades_periodicas,Propiedades periódicas
3,QUÍMICA. 2004. SEPTIEMBRE. EJERCICIO 2. OPCIÓN A,"Considere la serie de elementos: Li, Na, K, Rb...",propiedades_periodicas,Propiedades periódicas
4,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Propiedades periódicas
5,QUÍMICA. 2009. JUNIO. EJERCICIO 2. OPCIÓN A,La siguiente tabla proporciona los valores de ...,propiedades_periodicas,Comparación energías de ionización
6,QUÍMICA. 2013. JUNIO. EJERCICIO 2. OPCIÓN A,"Para los siguientes elementos Na, P, S y Cl, d...",propiedades_periodicas,Propiedades periódicas
7,QUÍMICA. 2013. RESERVA 3. EJERCICIO 2. OPCIÓN A,"Dado los elementos Cl, K y Ar, ordene razonada...",propiedades_periodicas,Propiedades periódicas
8,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,verdadero o falso
9,QUÍMICA. 2015. RESERVA 2. EJERCICIO 2. OPCIÓN A,Razone si las siguientes afirmaciones son verd...,propiedades_periodicas,Propiedades periódicas


In [ ]:
df.exam = df.exam.replace("QUIMICA", "QUÍMICA")